# Setup

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### IN Colab

In [12]:
#!unzip repo.zip -d /content/CSE151B_Kaggle

import os
os.chdir('/content/CSE151B_Kaggle')

In [13]:
!pip install prettyprint sympy numpy pandas matplotlib transformers accelerate vllm tqdm bitsandbytes ipykernel jupyter nvidia-nvjitlink

In [14]:
import os

# Point Colab's environment to the newly installed nvidia-nvjitlink package
lib_path = "/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib"

if "LD_LIBRARY_PATH" in os.environ:
    os.environ["LD_LIBRARY_PATH"] += f":{lib_path}"
else:
    os.environ["LD_LIBRARY_PATH"] = lib_path

In [ ]:
# At the end to download shit

!rm -rf /content/CSE151B_Kaggle/.venv

!zip -r CSE151B_Kaggle.zip /content/CSE151B_Kaggle

from google.colab import files
files.download('CSE151B_Kaggle.zip')


### Only if in DataHub

In [6]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

!export PATH="/home/ugheewala/.local/bin:$PATH"

# Create a virtual environment
!/home/ugheewala/.local/bin/uv venv .venv --seed --clear

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install prettyprint sympy numpy pandas matplotlib transformers accelerate vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/bin/bash: line 1: /home/ugheewala/.local/bin/uv: No such file or directory
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached vllm-0.20.0-cp38-abi3-manylinux_2_35_x86_64.whl.metadata (10 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached antlr4_python3_runtime-4.11.1-py3-none-any.whl.metadata (291 bytes)
ERROR: Operation cancelled by user
/content/CSE151B_Kaggle/.venv/bin/python: No module named ipykernel
Done. Restart the kernel befo

In [7]:
# !/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
#     "numpy<2" \
#     "torch==2.1.2+cu118" \
#     "transformers==4.51.3" \
#     "accelerate==0.34.2" \
#     "huggingface_hub>=0.23.0" \
#     "safetensors" \
#     "sentencepiece" \
#     "tqdm" \
#     "pandas" \
#     "matplotlib" \
#     "sympy" \
#     "antlr4-python3-runtime==4.11.1" \
#     --extra-index-url https://download.pytorch.org/whl/cu118

In [8]:
# !/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
#     "nvidia-cusparse-cu11" \
#     "nvidia-cublas-cu11" \
#     "nvidia-cuda-runtime-cu11" \
#     "nvidia-cudnn-cu11"

In [9]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

In [10]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "numpy<2" \
    "torch==2.3.1+cu121" \
    "torchvision==0.18.1+cu121" \
    "torchaudio==2.3.1+cu121" \
    --extra-index-url https://download.pytorch.org/whl/cu121

/bin/bash: line 1: name: No such file or directory


In [11]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "transformers==4.51.3" \
    "accelerate>=0.30.0" \
    "huggingface_hub>=0.23.0" \
    "safetensors" \
    "sentencepiece" \
    "tokenizers==0.21.4" \
    "sympy" \
    "pandas" \
    "matplotlib" \
    "tqdm" \
    "prettyprint" \
    "antlr4-python3-runtime==4.11.1" \
    "ipykernel" \
    "jupyter"

/bin/bash: line 1: name: No such file or directory


In [12]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "bitsandbytes==0.45.5"

/bin/bash: line 1: name: No such file or directory


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [31]:
import os
import sys
import json
import time
import csv
import subprocess
from pathlib import Path
from pprint import pprint

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
PUBLIC_DATA_PATH   = "data/public.jsonl"
PRIVATE_DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"

PROJECT_ROOT = Path.cwd()

RESULTS_DIR = PROJECT_ROOT / "results"
BASELINE1_DIR = RESULTS_DIR / "baseline1_weakest"
BASELINE1_DIR.mkdir(parents=True, exist_ok=True)

VAL_FRAC = 0.20
SPLIT_SEED = 414

CACHE_DIR = None
HF_HOME_DIR = None

MAX_INPUT_TOKENS = 4096
MAX_MODEL_LEN = 4096

MAX_NEW_TOKENS_SMOKE = 256
MAX_NEW_TOKENS_BASELINE = 1024

INFERENCE_BACKEND = "vllm"

BATCH_SIZE = 32
LOAD_IN_4BIT = True


MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

if HF_HOME_DIR is not None:
    os.environ["HF_HOME"] = str(HF_HOME_DIR)

if CACHE_DIR is not None:
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print("HF_HOME      :", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE :", os.environ.get("HF_HUB_CACHE"))
print("cache_dir    :", CACHE_DIR)

HF_HOME      : None
HF_HUB_CACHE : None
cache_dir    : None


In [16]:
import torch

print(f"CUDA_VISIBLE_DEVICES (Env): {os.environ.get('CUDA_VISIBLE_DEVICES')}")

cuda_available = torch.cuda.is_available()
print(f"Is CUDA available? {cuda_available}")

if cuda_available:
    print(f"Current Device: {torch.cuda.current_device()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("PyTorch still can't see the GPU.")
    device = torch.device("cpu")

CUDA_VISIBLE_DEVICES (Env): 0
Is CUDA available? True
Current Device: 0
Device Name: NVIDIA A100-SXM4-80GB


In [17]:
# import site

# roots = [Path(p) for p in site.getsitepackages()]
# matches = []

# for root in roots:
#     if root.exists():
#         matches.extend(root.rglob("libcusparse.so*"))

# for m in matches:
#     print(m)

In [18]:
# wanted_libs = {
#     "libcusparse.so",
#     "libcublas.so",
#     "libcudart.so",
#     "libcudnn.so",
# }

# lib_dirs = []

# for root in map(Path, site.getsitepackages()):
#     if not root.exists():
#         continue

#     for lib in wanted_libs:
#         for match in root.rglob(lib + "*"):
#             lib_dir = str(match.parent)
#             if lib_dir not in lib_dirs:
#                 lib_dirs.append(lib_dir)

# VENV = Path("/home/ugheewala/private/CSE151B_Kaggle/.venv")
# SITE = VENV / "lib/python3.11/site-packages"

# cuda11_dirs = [
#     SITE / "nvidia/cusparse/lib",
#     SITE / "nvidia/cublas/lib",
#     SITE / "nvidia/cuda_runtime/lib",
#     SITE / "nvidia/cudnn/lib",
#     SITE / "torch/lib",
# ]

# cuda11_dirs = [str(p) for p in cuda11_dirs if p.exists()]

# path_line = ":".join(cuda11_dirs)

# print("Add this before starting the notebook/kernel:")
# print(f'export LD_LIBRARY_PATH="{path_line}:$LD_LIBRARY_PATH"')

In [19]:
# import os
# import subprocess
# from pathlib import Path

# VENV = Path("/home/ugheewala/private/CSE151B_Kaggle/.venv")
# SITE = VENV / "lib/python3.11/site-packages"

# cuda11_dirs = [
#     SITE / "nvidia/cusparse/lib",
#     SITE / "nvidia/cublas/lib",
#     SITE / "nvidia/cuda_runtime/lib",
#     SITE / "nvidia/cudnn/lib",
#     SITE / "torch/lib",
# ]

# env = os.environ.copy()
# env["LD_LIBRARY_PATH"] = ":".join(str(p) for p in cuda11_dirs if p.exists()) + ":" + env.get("LD_LIBRARY_PATH", "")

# subprocess.run(
#     [str(VENV / "bin/python"), "-m", "bitsandbytes"],
#     env=env,
# )

In [20]:
# import json
# from pathlib import Path

# kernel_json = Path("/home/ugheewala/.local/share/jupyter/kernels/cse151b/kernel.json")

# with open(kernel_json, "r") as f:
#     spec = json.load(f)

# ld_library_path = (
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cusparse/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cublas/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cudnn/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/torch/lib:"
#     "${LD_LIBRARY_PATH}"
# )

# spec.setdefault("env", {})
# spec["env"]["LD_LIBRARY_PATH"] = ld_library_path
# spec["env"]["BNB_CUDA_VERSION"] = "118"

# with open(kernel_json, "w") as f:
#     json.dump(spec, f, indent=2)

# print(kernel_json)
# print(json.dumps(spec, indent=2))

In [21]:
import transformers

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

torch: 2.11.0+cu130
cuda: 13.0
cuda available: True
transformers: 5.7.0


In [22]:
try:
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
except Exception as e:
    print("bitsandbytes import failed:", repr(e))

bitsandbytes: 0.49.2


In [23]:
from transformers.utils import is_torch_available, is_bitsandbytes_available

print("is_torch_available:", is_torch_available())
print("is_bitsandbytes_available:", is_bitsandbytes_available())

is_torch_available: True
is_bitsandbytes_available: True


In [24]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

from baseline.datasets import load_public_splits, load_private_set
from baseline.generation import GenerationConfig
from baseline.prompt_sets import build_prompt_texts
from baseline.modeling import ModelConfig, load_transformers_model, predownload_model, load_model, detect_gpu_info
from baseline.scoring import load_judger, score_one, summarize_results
from baseline.progress_viz import RunProgressDashboard
from prompting.prompt_chain import build_prompt_chain
from baseline.runner import run_problem_set, benchmark_batch_sizes

In [25]:
gpu_info = detect_gpu_info()
pprint(gpu_info.to_dict())

{'capability': (8, 0),
 'cuda_available': True,
 'device_count': 1,
 'device_name': 'NVIDIA A100-SXM4-80GB'}


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices - present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [26]:
splits = load_public_splits(PUBLIC_DATA_PATH, val_frac=VAL_FRAC, seed=SPLIT_SEED)

train_set = splits["train"]
val_set = splits["val"]
public_set = splits["public"]
private_set = load_private_set(PRIVATE_DATA_PATH)

print("Train summary:")
pprint(train_set.summary())

print("\nValidation summary:")
pprint(val_set.summary())

print("\nPublic summary:")
pprint(public_set.summary())

print("\nPrivate summary:")
pprint(private_set.summary())

Train summary:
{'n': 901,
 'n_answered': 901,
 'n_free_form': 601,
 'n_mcq': 300,
 'name': 'public_train'}

Validation summary:
{'n': 225,
 'n_answered': 225,
 'n_free_form': 150,
 'n_mcq': 75,
 'name': 'public_val'}

Public summary:
{'n': 1126,
 'n_answered': 1126,
 'n_free_form': 751,
 'n_mcq': 375,
 'name': 'public'}

Private summary:
{'n': 943, 'n_answered': 0, 'n_free_form': 643, 'n_mcq': 300, 'name': 'private'}


In [27]:
prompt_chain = build_prompt_chain(strategy_name="baseline")

for label, problem_set in [("train", train_set), ("val", val_set), ("private", private_set)]:
    problem = problem_set.problems()[0]
    spec = prompt_chain.build_spec(problem)

    print("=" * 80)
    print(label, "id=", problem.id, "template=", spec.name)
    print("metadata:", spec.metadata)
    print("generation_hints:", spec.generation_hints)
    print(spec.to_messages()[0]["content"][:300])
    print("--- user ---")
    print(spec.to_messages()[-1]["content"][:500])

train id= 499 template= baseline_mcq
metadata: {'strategy_name': 'baseline', 'route_name': 'mcq', 'tags': []}
generation_hints: {'temperature': 0.6, 'top_p': 0.95}
You are an expert mathematician. Read the problem and the answer choices below, then select the single best answer. Output only the letter of your chosen option inside \boxed{}, e.g. \boxed{C}.
--- user ---
We now define an algorithm: The definition of a(n) is the least odd number k such that k * 2^n + 1 is a prime number. Given the input x_list (a series of values): [70, 71, 72, 73, 74, 75, 76, 77, 78, 79], determine the corresponding output sequence y_list.

Answer choices:
A. [44, 43, 129, 26, 63, 1, 90, 33, 22, 243]
B. [37, 35, 122, 19, 64, 10, 96, 26, 20, 245]
C. [38, 40, 128, 22, 71, 3, 91, 28, 14, 248]
D. [43, 37, 125, 21, 70, 9, 98, 27, 13, 246]
E. [39, 39, 127, 23, 67, 5, 93, 29, 15, 249]

val id= 990 template= baseline_free_form
metadata: {'strategy_name': 'baseline', 'route_name': 'free_form', 'tags': []}
generati

## 4. Modeling

In [28]:
model_config = ModelConfig(
    model_id=MODEL_ID,
    backend=INFERENCE_BACKEND,
    cache_dir=CACHE_DIR,
    gpu_id=GPU_ID,
    max_input_tokens=MAX_INPUT_TOKENS,
    max_model_len=MAX_MODEL_LEN,
    dtype="bfloat16",
    torch_dtype="bfloat16",
    load_in_4bit=False,
    device_map="auto",
    low_cpu_mem_usage=True,
    gpu_memory_utilization=0.85,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
    reuse_loaded=True,
)

t0 = time.perf_counter()
model_bundle = load_model(model_config)
print(f"Model load/reuse time: {time.perf_counter() - t0:.2f} sec")
print("Backend:", model_bundle.backend)
print("Device:", model_bundle.device())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 05-01 04:46:08 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 4096, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-01 04:46:26 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-01 04:46:26 [nixl_utils.py:34] NIXL is not available
WARNING 05-01 04:46:26 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-01 04:46:26 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-01 04:46:26 [model.py:1680] Using max model len 4096
INFO 05-01 04:46:26 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 05-01 04:46:26 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-01 04:46:26 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfi

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

WARNING 05-01 04:46:29 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
Model load/reuse time: 110.03 sec
Backend: vllm
Device: vllm


# Baseline 1: Naive

In [29]:
baseline1_prompt_chain = build_prompt_chain(strategy_name="baseline")

smoke_generation_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS_SMOKE,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

baseline1_generation_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS_BASELINE,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

In [30]:
batch_report = benchmark_batch_sizes(
    problem_set=train_set,
    model_bundle=model_bundle,
    batch_sizes=[1, 2, 4, 8, 16, 32, 64],
    prompt_chain=baseline1_prompt_chain,
    generation_config=smoke_generation_config,
    sample_size=16,
    score=False,
    show_progress=False,
)

pd.DataFrame(batch_report["rows"])

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

,batch_size,ok,elapsed_sec,sec_per_problem,generation_sec_per_problem,error
0,1,True,31.408623,1.963039,1.961678,None
1,2,True,16.194406,1.012150,1.012045,None
2,4,True,8.465179,0.529074,0.528974,None
3,8,True,13.673946,0.854622,0.854519,None
4,16,True,2.361134,0.147571,0.147469,None
5,32,True,2.359985,0.147499,0.147399,None
6,64,True,2.360521,0.147533,0.147428,None


In [ ]:
# Train split smoke run

baseline1_train_result = run_problem_set(
    problem_set=train_set,
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=32,
    score=True,
    output_jsonl_path=BASELINE1_DIR / "train_results.jsonl",
    report_json_path=BASELINE1_DIR / "train_report.json",
)

pprint(baseline1_train_result.report)

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.3076923076923077,
             'mcq_acc': 0.05263157894736842,
             'n_correct': 5,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.15625},
 'timings': {'generation_sec': 11.5697886199996,
             'prompt_build_sec': 0.002926810000644764,
             'scoring_sec': 0.7455679300001066}}
"""

In [ ]:
# Validation run

baseline1_val_result = run_problem_set(
    problem_set=val_set,
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=True,
    output_jsonl_path=BASELINE1_DIR / "val_results.jsonl",
    report_json_path=BASELINE1_DIR / "val_report.json",
)

pprint(baseline1_val_result.report)

In [36]:
"""
 'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.12666666666666668,
             'mcq_acc': 0.08,
             'n_correct': 25,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.1111111111111111},
 'timings': {'generation_sec': 87.61873525499959,
             'prompt_build_sec': 0.01957373799996276,
             'scoring_sec': 7.864173332000064}}
"""

"\n 'generation_config': {'do_sample': True,\n                       'max_new_tokens': 1024,\n                       'min_p': 0.0,\n                       'presence_penalty': 0.0,\n                       'repetition_penalty': 1.0,\n                       'temperature': 0.6,\n                       'top_k': 20,\n                       'top_p': 0.95},\n 'output_jsonl_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/val_results.jsonl',\n 'problem_set': {'n': 225,\n                 'n_answered': 225,\n                 'n_free_form': 150,\n                 'n_mcq': 75,\n                 'name': 'public_val'},\n 'score_available': True,\n 'submission_csv_path': None,\n 'summary': {'free_form_acc': 0.12666666666666668,\n             'mcq_acc': 0.08,\n             'n_correct': 25,\n             'n_free_form': 150,\n             'n_mcq': 75,\n             'n_scored': 225,\n             'overall_acc': 0.1111111111111111},\n 'timings': {'generation_sec': 87.61873525499959,\n             

In [ ]:
# Test run

PRIVATE_LIMIT = None

baseline1_private_result = run_problem_set(
    problem_set=private_set,
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=False,
    output_jsonl_path=BASELINE1_DIR / "private_results.jsonl",
    submission_csv_path=BASELINE1_DIR / "submission.csv",
    report_json_path=BASELINE1_DIR / "private_report.json",
)

pprint(baseline1_private_result.report)

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 339.3967333410001,
             'prompt_build_sec': 0.07438255999932153,
             'scoring_sec': 0.0008840780001264648}}
"""

In [35]:
submission_path = BASELINE1_DIR / "submission.csv"

if submission_path.exists():
    sub_df = pd.read_csv(submission_path)
    print(sub_df.shape)
    display(sub_df.head())
    print("Columns:", list(sub_df.columns))
else:
    print("No submission file yet. Run the private test cell first.")

(943, 2)


,id,response
0,0,"Okay, let's tackle part a) first. The problem ..."
1,1,"Okay, let's try to figure out this problem. So..."
2,2,"Okay, let's tackle this problem step by step. ..."
3,3,"Okay, let's try to figure out this problem ste..."
4,4,"Okay, let's see. The problem says that the poi..."


Columns: ['id', 'response']


# Baseline 2: Output hardening

In [ ]:
from baseline.baseline2_runner import run_baseline2_problem_set
from baseline.generation import GenerationConfig

BASELINE2_DIR = RESULTS_DIR / "baseline2_prompt_format"
BASELINE2_DIR.mkdir(parents=True, exist_ok=True)

baseline2_generation_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS_BASELINE,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

In [ ]:
baseline2_train_result = run_baseline2_problem_set(
    problem_set=train_set,
    model_bundle=model_bundle,
    generation_config=baseline2_generation_config,
    batch_size=BATCH_SIZE,
    limit=32,
    score=True,
    output_jsonl_path=BASELINE2_DIR / "train_results.jsonl",
    debug_jsonl_path=BASELINE2_DIR / "train_debug.jsonl",
    report_json_path=BASELINE2_DIR / "train_report.json",
    show_progress=True,
)

pprint(baseline2_train_result.report)

In [ ]:
baseline2_val_result = run_baseline2_problem_set(
    problem_set=val_set,
    model_bundle=model_bundle,
    generation_config=baseline2_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=True,
    output_jsonl_path=BASELINE2_DIR / "val_results.jsonl",
    debug_jsonl_path=BASELINE2_DIR / "val_debug.jsonl",
    report_json_path=BASELINE2_DIR / "val_report.json",
    show_progress=True,
)

pprint(baseline2_val_result.report)

In [ ]:
RUN_BASELINE2_PRIVATE = False

if RUN_BASELINE2_PRIVATE:
    baseline2_private_result = run_baseline2_problem_set(
        problem_set=private_set,
        model_bundle=model_bundle,
        generation_config=baseline2_generation_config,
        batch_size=BATCH_SIZE,
        limit=None,
        score=False,
        output_jsonl_path=BASELINE2_DIR / "private_results.jsonl",
        debug_jsonl_path=BASELINE2_DIR / "private_debug.jsonl",
        submission_csv_path=BASELINE2_DIR / "submission.csv",
        report_json_path=BASELINE2_DIR / "private_report.json",
        show_progress=True,
    )

    pprint(baseline2_private_result.report)

In [ ]:
pd.DataFrame([
    baseline2_val_result.report["formatting"]
]).T.rename(columns={0: "value"})

In [ ]:
baseline2_val_result.report["formatting"]["schema_error_counts"]

In [ ]:
bad_schema_rows = [
    row for row in baseline2_val_result.scored_rows
    if not row.get("schema_valid")
]

len(bad_schema_rows), bad_schema_rows[:3]

In [ ]:
comparison_rows = []

if "baseline1_val_result" in globals():
    comparison_rows.append({
        "baseline": "baseline1",
        **baseline1_val_result.report["summary"],
    })

comparison_rows.append({
    "baseline": "baseline2",
    **baseline2_val_result.report["summary"],
    "schema_valid_rate": baseline2_val_result.report["formatting"]["schema_valid_rate"],
    "extractable_rate": baseline2_val_result.report["formatting"]["extractable_rate"],
    "retry_rate": baseline2_val_result.report["formatting"]["retry_rate"],
})

pd.DataFrame(comparison_rows)

# Baseline 3: Prompt Engineering

# Baseline 4: Supervised Fine-Tuning (SFT)

## Post-SFT Tuning

# Baseline 5: RL